# <img style="float: left; padding-right: 10px; width: 100px" src="https://raw.githubusercontent.com/sbildea/SEED/main/imagini/unitbv_mie.png"> <center>Știința Explorării și Exploatării Datelor - SEED1</center>


##    <center>Curs 7: De la analiză la recomandare</center>
**Universitatea Transilvania**<br>
**SIIPA, Semestrul I**<br>
**Instructor:** Conf. dr. Teodor &Scedil;tefan B&icirc;ldea<br>
**Contact:** teodor.bildea@unitbv.ro<br>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sbildea/SEED/blob/main/cursuri/07-proiect/curs-07-de-la-analiza-la-recomandare.ipynb)

In [ ]:
# Configurare curs SEED: rulați această celulă prima.
# Colab: descarcă materialele cursului și instalează pachetele lipsă (aprox. un minut).
# Local: găsește singură rădăcina repository-ului, oricare ar fi folderul curent al kernelului.
import os, sys
MODUL = "07-proiect"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/SEED"):
        !git clone -q --depth 1 https://github.com/sbildea/SEED.git /content/SEED
        !pip install -q -r /content/SEED/requirements.txt
    RADACINA = "/content/SEED"
else:
    RADACINA = os.path.abspath(".")
    while not os.path.isdir(os.path.join(RADACINA, "seed")) and os.path.dirname(RADACINA) != RADACINA:
        RADACINA = os.path.dirname(RADACINA)
    import importlib.util, socket
    lipsa = [p for p in ("pandas", "matplotlib", "seaborn", "sklearn") if importlib.util.find_spec(p) is None]
    if lipsa:
        print("Lipsesc pachetele", lipsa, "- în terminal: pip install -r requirements-local.txt")
    try:
        socket.create_connection(("raw.githubusercontent.com", 443), 2).close()
    except OSError:
        print("Fără internet: imaginile din text nu se văd. O singură dată, în terminal: python tools/imagini_offline.py")
os.chdir(os.path.join(RADACINA, "cursuri", MODUL))
sys.path.insert(0, RADACINA)

## Ce învățăm azi

Până acum am învățat **metode**. În practică, un analist sau un consultant este plătit pentru **decizii mai bune**, nu pentru modele. Azi legăm totul într-un flux de lucru pe care îl veți aplica în proiect:

1. de la cererea vagă a clientului la o **întrebare la care datele pot răspunde**;
2. **fluxul complet** al unui proiect de date (CRISP-DM);
3. cum arată un **raport** pentru cineva care nu știe statistică;
4. ce se întâmplă **după** raport: implementare, monitorizare, etică.

## 1. De la cererea clientului la întrebarea analitică

Clienții nu vin cu întrebări de *machine learning*, ci cu probleme de afaceri. Prima voastră sarcină este traducerea:

| Ce spune clientul | Întrebarea analitică | Tipul problemei (modulul) | Cum măsurăm succesul |
|---|---|---|---|
| „Pierdem prea mulți abonați.” | Care clienți vor pleca în luna următoare? | clasificare (5, 6) | profitul campaniei de retenție, recall |
| „Nu știu cât să cer pe o casă.” | Ce preț are o locuință cu aceste caracteristici? | regresie (4) | eroarea medie în lei (MAE) |
| „Vreau oferte personalizate.” | Ce grupuri de clienți au comportament asemănător? | clusterizare (3) | grupuri stabile și interpretabile |
| „Ce produse să pun alături?” | Ce produse se cumpără împreună? | reguli de asociere (3) | lift, creșterea coșului mediu |
| „Ce se întâmplă cu vânzările?” | Cum au evoluat vânzările pe produse și regiuni? | analiză exploratorie (2) | un raport care răspunde la întrebare |

O întrebare bună este **specifică** (cine, ce, când), **măsurabilă** și are o **decizie** legată de ea: *dacă răspunsul este X, facem Y*.

**Exercițiu (5 minute):** reformulați ca întrebare analitică: *„Vrem să reducem stocurile fără să pierdem vânzări.”*

## 2. Fluxul complet: CRISP-DM

CRISP-DM (*Cross-Industry Standard Process for Data Mining*) este cel mai folosit cadru pentru proiectele de date. Pașii nu sunt liniari: ne întoarcem des la pasul anterior.

| Pas | Întrebarea pasului | Unde l-am învățat |
|---|---|---|
| 1. Înțelegerea problemei | Ce decizie trebuie luată și cum măsurăm succesul? | modulul 1, acest curs |
| 2. Înțelegerea datelor | Ce date avem, ce lipsește, ce e ciudat? | modulul 2 |
| 3. Pregătirea datelor | Ce curățăm, ce transformăm, ce coloane folosim? | modulele 2-3 |
| 4. Modelare | Ce metode se potrivesc întrebării? | modulele 3-5 |
| 5. Evaluare | Este modelul suficient de bun **pentru această decizie**? | modulul 6 |
| 6. Implementare | Cum ajunge rezultatul la cei care decid, și cum îl menținem? | acest curs |

**Roluri:** *analistul de date* duce pașii 2-5; *consultantul* răspunde de pașii 1 și 5 (și de comunicare); *consultantul de implementare* duce pasul 6, adesea într-un ERP sau CRM. Toți trebuie să înțeleagă întregul flux.

## 3. Raportul: pentru cine nu știe statistică

Managerul citește primul paragraf și, poate, un grafic. Începeți cu **concluzia** (principiul piramidei), apoi argumentele:

1. **Recomandarea**, într-o frază, cu o cifră: *„Trimiteți oferta celor 520 de clienți marcați; profit estimat 60.000 lei pe lună.”*
2. **Contextul:** întrebarea și datele (perioadă, câte înregistrări, de unde provin).
3. **Dovezile:** 2-3 rezultate cheie, fiecare cu un grafic simplu și o propoziție de interpretare.
4. **Cât de sigur este:** performanța pe date noi (test), comparată cu un model naiv.
5. **Limitele și riscurile:** ce nu poate spune analiza, ipotezele făcute.
6. **Pașii următori:** ce decizie, cine o ia, cum verificăm efectul.

**Reguli de scriere:**

- o cifră fără unitate și fără comparație nu spune nimic („acuratețe 80%” → „găsim 4 din 5 clienți care pleacă, față de 0 fără model”);
- graficele au titlu care spune concluzia („Clienții cu contract lunar pleacă de 3 ori mai des”), nu doar „Grafic churn”;
- fără termeni tehnici nedefiniți; dacă e nevoie de unul, explicați-l într-o paranteză.

## 4. Lucrul cu asistentul AI la proiect

Asistentul vă ajută să scrieți cod și text mai repede, dar **responsabilitatea rezultatelor este a voastră**. La proiect:

- [ ] notebook-ul rulează complet după **Restart and run all**, cu `random_state` fixat;
- [ ] fiecare cifră din raport apare în rezultatele notebook-ului (asistentul poate „inventa” cifre plauzibile);
- [ ] puteți explica oral fiecare pas și de ce l-ați ales;
- [ ] completați **Jurnalul AI**: cererile importante, ce ați verificat, ce ați corectat.

## 5. După raport: implementare și monitorizare

Un model care rămâne într-un notebook nu produce valoare. Întrebări pe care le pune un consultant de implementare:

- **Unde ajunge rezultatul?** O listă exportată lunar, un câmp nou în CRM, un raport în Power BI, o regulă în ERP.
- **Cât de des se recalculează?** Zilnic, lunar, la cerere.
- **Cum aflăm că s-a stricat?** Comportamentul clienților se schimbă (*data drift*): comparăm lunar performanța pe date noi cu cea de la test.
- **Cine răspunde?** Cine reantrenează modelul și cine aprobă schimbările.

## 6. Etică și date personale

- **GDPR:** folosim doar datele necesare scopului; datele personale nu se trimit în asistenți AI publici.
- **Discriminare:** un model poate reproduce nedreptăți din datele istorice (de exemplu după vârstă, gen sau zonă). Verificați performanța pe grupuri.
- **Explicabilitate:** pentru decizii care afectează oameni (credit, angajare), un model simplu, explicabil, este adesea preferat unuia puțin mai precis.

## Puntea curs → seminar

**Concepte cheie**

1. **Fluxul complet:** întrebare → date → explorare → pregătire → model → evaluare → decizie.
2. **Reproductibilitate:** notebook-ul rulează de la început la sfârșit, cu `random_state` fixat.
3. **Comunicarea rezultatelor:** pentru cine nu știe statistică; cifre, nu impresii.
4. **Limitele modelului:** ce nu poate spune și unde poate greși.

**Ce ceri asistentului AI** (exemplu de cerere)

> Pe baza rezultatelor din notebook-ul meu, scrie un rezumat de 5 propoziții pentru un manager: întrebarea, datele, modelul ales, cât de bun este pe datele de test și ce decizie recomand. Menționează o limită a analizei.

**Ce verifici înainte să accepți rezultatul**

- [ ] fiecare cifră din rezumat apare în rezultatele notebook-ului;
- [ ] notebook-ul rulează complet după **Runtime → Restart and run all**;
- [ ] limitele sunt menționate cinstit (date, metrică, generalizare).

Continuăm la seminar: [seminar-07-de-la-intrebare-la-decizie.ipynb](seminar-07-de-la-intrebare-la-decizie.ipynb).